In [3]:
!pip install supabase


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.0/48.0 kB 2.7 MB/s eta 0:00:00


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
import json
import psycopg2
from sentence_transformers import SentenceTransformer

Load embedding model

In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Import environment variables

In [7]:
import os
from dotenv import load_dotenv

load_dotenv('/content/drive/MyDrive/SpringForge/.env')

url = os.getenv("SUPABASE_URL")
key = os.getenv("SUPABASE_KEY")

if not key:
    raise ValueError("Supabase Key not found. Check your .env file.")

Connect to Postgress

In [8]:
from supabase import create_client

supabase = create_client(url, key)

Load dataset

In [9]:
import json

default_path = "/content/drive/MyDrive/SpringForge/normalized_dataset.json"
dataset_path = os.environ.get("DATASET_PATH", default_path)

with open(dataset_path) as f:
    dataset = json.load(f)

len(dataset)

232

Generate Embeddings

In [ ]:
def generate_embedding(item):
    text = item["title"] + "\n" + item["content"]
    return model.encode(text).tolist()

Insert rows into Supabase

In [ ]:
from tqdm import tqdm

BATCH_SIZE = 50
batch = []

for item in tqdm(dataset):
    emb = generate_embedding(item)

    row = {
        "id": item["id"],
        "source": item["source"],
        "url": item.get("url"),
        "title": item.get("title"),
        "content": item.get("content"),
        "tags": item.get("tags", []),
        "chunk_id": item.get("chunk_id"),
        "embedding": emb
    }

    batch.append(row)

    if len(batch) == BATCH_SIZE:
        supabase.table("knowledge_base").insert(batch).execute()
        batch = []


if batch:
    supabase.table("knowledge_base").insert(batch).execute()

print("All embeddings generated + inserted successfully!")

100%|██████████| 232/232 [00:36<00:00,  6.35it/s]


All embeddings generated + inserted successfully!
